In [5]:
import seaborn as sns
import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs
import altair as alt
import plotly.express as px
import plotly.graph_objects as go
import great_tables as tg
import datetime as dt
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE

from Kaggle.Bank_Transaction_Fraud_Detection import random_forest

In [6]:
df_path = r"F:\Datasets\CSV datasets\clasification\Airlines.csv"

In [7]:
df = pl.read_csv(df_path)

In [16]:
df.null_count()

Airline,Flight,AirportFrom,AirportTo,DayOfWeek,Time,Length,Delay
u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0


In [8]:
df

id,Airline,Flight,AirportFrom,AirportTo,DayOfWeek,Time,Length,Delay
i64,str,i64,str,str,i64,i64,i64,i64
1,"""CO""",269,"""SFO""","""IAH""",3,15,205,1
2,"""US""",1558,"""PHX""","""CLT""",3,15,222,1
3,"""AA""",2400,"""LAX""","""DFW""",3,20,165,1
4,"""AA""",2466,"""SFO""","""DFW""",3,20,195,1
5,"""AS""",108,"""ANC""","""SEA""",3,30,202,0
…,…,…,…,…,…,…,…,…
539379,"""CO""",178,"""OGG""","""SNA""",5,1439,326,0
539380,"""FL""",398,"""SEA""","""ATL""",5,1439,305,0
539381,"""FL""",609,"""SFO""","""MKE""",5,1439,255,0


In [9]:
df = df.drop('id')

In [11]:
X = df.select(cs.exclude('Delay'))

In [12]:
y = df.get_column('Delay')

In [13]:
from sklearn.model_selection import train_test_split

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [33]:
from sklearn.feature_extraction import FeatureHasher
from sklearn.impute import SimpleImputer

In [31]:
categorical_cols = X.select(cs.string()).columns
numerical_cols = X.select(cs.numeric()).columns

In [36]:
imputer_categorical = SimpleImputer(strategy='most_frequent')
X_train[categorical_cols] = imputer_categorical.fit_transform(X_train[categorical_cols])
X_test[categorical_cols] = imputer_categorical.transform(X_test[categorical_cols])

In [38]:
X_train_categorical_pandas = pd.DataFrame(X_train[categorical_cols], columns=categorical_cols)
X_test_categorical_pandas = pd.DataFrame(X_test[categorical_cols], columns=categorical_cols)

In [39]:
hasher = FeatureHasher(n_features=32, input_type='string')

In [45]:
hashed_features_train = hasher.fit_transform(X_train_categorical_pandas.values.tolist())
hashed_features_test = hasher.transform(X_test_categorical_pandas.values.tolist())

In [40]:
X_train_numeric_pandas = pd.DataFrame(X_train[numerical_cols], columns=numerical_cols)
X_test_numeric_pandas = pd.DataFrame(X_test[numerical_cols], columns = numerical_cols)

In [41]:
from sklearn.preprocessing import StandardScaler

In [42]:
scaler = StandardScaler()

In [43]:
scaled_numerical_train = scaler.fit_transform(X_train_numeric_pandas)
scaled_numerical_test = scaler.transform(X_test_numeric_pandas)

In [46]:
X_train_hashed = np.hstack([hashed_features_train.toarray(), scaled_numerical_train])
X_test_hashed = np.hstack([hashed_features_test.toarray(), scaled_numerical_test])

In [58]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

In [56]:
random_forest = RandomForestClassifier()

In [57]:
param_grid = {
      'n_estimators': [10, 25, 50, 100, 150, 200, 250, 500, 1000],
      'max_depth': [None, 10, 20, 30],
      'min_samples_split': [2, 5, 10],
      'min_samples_leaf': [1, 2, 4],
      'max_features': [1.0, 'sqrt', 'log2'],
      'criterion': ['gini', 'entropy'],
      'bootstrap': [True, False]
    }

In [59]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [63]:
grid_search = GridSearchCV(
    estimator=random_forest,
    param_grid=param_grid,
    scoring='accuracy',
    cv=cv,
    n_jobs=8,
    verbose=2,
    return_train_score=True
)

In [64]:
grid_search.fit(hashed_features_test, y_test)

Fitting 5 folds for each of 3888 candidates, totalling 19440 fits


KeyboardInterrupt: 